# Experiment 8 — Bottleneck localization: is the law represented, and does the output get it right?

**Question.** Experiment 3 found that law identity is recoverable from mid-stack activations
(87% cross-theme retrieval at `hidden_states[24]`, last token). That is correlational: the
representation exists, but we do not know whether the model *uses* it. This notebook takes the
first step by crossing the two axes on the same examples:

- **axis 1 — is the law represented?** Read the story's activation at the final prompt token,
  before any generation, and ask whether its nearest neighbour among stories of *other themes*
  encodes the same law (experiment 3's metric, per example rather than aggregated).
- **axis 2 — did the model formalize it correctly?** Generate Rigid Grammar in no-think mode
  from the same prompt and grade it with `checkform`.

Crossing them localizes the failure, which is Ke's question and step 5 of the experiment-7
proposal:

| | correct output | wrong output |
|---|---|---|
| **law represented** | the pipeline works | **formalization bottleneck** — the model knows which law it is and still emits the wrong grammar |
| **law not represented** | got lucky / shallow route | **abstraction bottleneck** — the law never made it into the representation |

Whichever failure cell dominates says where the no-think gap lives, and therefore what
experiment 1's steering should target.

**Design notes.**

- Both axes are computed from the **same input** — the chat-formatted formalization prompt — so
  the representation and the behaviour cannot be blamed on different conditioning. Experiment 3
  used bare texts, which is why activations are re-extracted here rather than reused.
- Only stories are used (four themes per law), so the retrieval pool is apples-to-apples: every
  candidate shares the prompt template and differs in theme, and themes have near-disjoint
  vocabulary by construction.
- **Complexity is the obvious confound**: both axes degrade as laws get bigger, so a raw
  association could be entirely driven by operation count. Every headline number is therefore
  also reported *within* complexity bins.

**Model.** `Qwen/Qwen3-4B`, no-think, vendor-recommended sampling — same as experiments 1 and 3.

**Runtime.** Colab GPU. Unlike experiment 3 this notebook generates (~200 prompts × 256 tokens),
so budget ~10–20 min on a T4; activations and graded rows are both checkpointed to Drive, and
the analysis cells then re-run on CPU.

In [ ]:
# ------------------------------ Configuration ------------------------------

REPO_URL = "https://github.com/shivam-raval96/semantic-drift-autoformalization.git"
REPO_COMMIT = "c9e128f247b6daff5b47fd9711ded5e02d1095e9"  # same pin as experiments 01 and 03

MODEL_NAME = "Qwen/Qwen3-4B"

SEED = 0

# Same sampler and size as experiment 03, so the pair set is identical.
PER_BIN = 10
STORY_THEMES = ["graft", "paint", "signal", "tea"]

# Residual-stream read points (hidden_states indices). PRIMARY_LAYER is
# experiment 03's retrieval peak and carries the headline cross-tab; the others
# are swept to check the conclusion is not layer-specific.
LAYERS = [12, 18, 24, 30]
PRIMARY_LAYER = 24

MAX_NEW_TOKENS = 256
BATCH_SIZE = 8

# Smoke-test mode: a handful of prompts, to validate generation and grading
# plumbing before committing to the full run.
QUICK_TEST = False
if QUICK_TEST:
    PER_BIN, STORY_THEMES, LAYERS = 2, ["paint", "tea"], [12, 24]
    PRIMARY_LAYER = 24

In [ ]:
# ------------------------- Environment and outputs -------------------------

%pip install -q -U "transformers>=4.51" accelerate matplotlib

import hashlib
import json
import subprocess
import sys
from pathlib import Path

if not Path("semantic-drift-autoformalization").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
subprocess.run(
    ["git", "-C", "semantic-drift-autoformalization", "checkout", "-q", REPO_COMMIT],
    check=True,
)
sys.path.insert(0, str(Path("semantic-drift-autoformalization/informalizing-etp").resolve()))

from benchmark import drop_vacuous, load_equations, sample_pairs_stratified, wrap_prompt
from checkform import build_prompt, grade
from storyform import render_story

import numpy as np
import torch
from tqdm.auto import tqdm

OUT_DIR = Path("exp08-outputs")
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/mech-interp-experiments/exp08-bottleneck")
except Exception as error:
    print(f"Google Drive not available ({error}); using local {OUT_DIR}/")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR.resolve())

## Dataset — the same laws, four themed stories each, as formalization prompts

Same seeded stratified sampler and `PER_BIN` as experiment 3, so the pair set matches. Each law
is rendered as four themed stories (deterministic, no LLM), and each story is wrapped in the
repo's own formalization prompt (`formalize_prompt.md` via `build_prompt`, plus the benchmark's
uniform no-think regime wording via `wrap_prompt`) — byte-for-byte the input experiment 1 grades.

Every record therefore carries what both axes need: the exact prompt text (for the forward pass
and for generation) and the pair's canonical metadata (for grading).

In [ ]:
# ------------------------------ Build the dataset ------------------------------

equations, equations_sha = load_equations()
print(f"ETP equation list: {len(equations)} equations (sha256 {equations_sha[:12]})")

pairs = drop_vacuous(sample_pairs_stratified(equations, PER_BIN, SEED, form="story"))
print(f"{len(pairs)} pairs after dropping vacuous laws")

records = []
for sample in pairs:
    meta = sample["metadata"]
    for theme in STORY_THEMES:
        story, story_meta = render_story(meta["equation_e"], meta["equation_f"], theme_key=theme)
        records.append({
            "pair_id": sample["pair_id"],
            "theme": theme,
            "ops_total": sample["ops_total"],
            "depth": sample["depth"],
            "metadata": story_meta,
            # Empty model arg: no-think is switched via the chat template, not /no_think.
            "prompt": wrap_prompt(build_prompt({"story": story}), "off", "", "story"),
        })

print(f"{len(pairs)} pairs x {len(STORY_THEMES)} themes = {len(records)} prompts")
print(f"\n--- prompt tail for {records[0]['pair_id']} ({records[0]['theme']}) ---")
print("..." + records[0]["prompt"][-600:])

## Read the representation, then generate from it

Two passes over the identical chat-formatted prompt:

1. **Read** — one forward pass with `output_hidden_states`, keeping the activation at the **final
   prompt token**. That is the pre-generation "summary" position: everything the model has
   understood about the problem, just before it starts answering.
2. **Generate** — no-think decoding with Qwen3's vendor-recommended sampling (as in experiment
   1), graded by `checkform.grade` into `correct` / `wrong` / `unparseable`.

Both are checkpointed (`activations.pt`, `graded.jsonl`) and reloaded on rerun, so an interrupted
session resumes and the analysis cells never need the GPU. The model is only loaded if one of
the two caches is missing.

In [ ]:
# -------------------- Activations and graded generations --------------------

RUN_SHA = hashlib.sha256(
    json.dumps([MODEL_NAME, LAYERS, MAX_NEW_TOKENS, SEED,
                [r["prompt"] for r in records]]).encode()).hexdigest()[:12]
ACT_CACHE = OUT_DIR / f"activations-{RUN_SHA}.pt"
ROWS_PATH = OUT_DIR / f"graded-{RUN_SHA}.jsonl"


def load_rows(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


rows = load_rows(ROWS_PATH)
if len(rows) != len(records):
    rows = []  # partial or stale file: regenerate from scratch

if ACT_CACHE.exists() and rows:
    acts = torch.load(ACT_CACHE)
    print(f"loaded cached activations and {len(rows)} graded rows")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    else:
        dtype = torch.float32
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map="auto")
    model.eval()
    assert max(LAYERS) <= model.config.num_hidden_layers

    def build_chat(prompt_text):
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt_text}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)

    chats = [build_chat(r["prompt"]) for r in records]

    @torch.no_grad()
    def collect_final_token(chats, layers):
        """Activation at the last prompt token, per requested hidden_states index."""
        tokenizer.padding_side = "right"  # keeps last-token indexing by length valid
        out = {L: [] for L in layers}
        for i in tqdm(range(0, len(chats), BATCH_SIZE), desc="activations"):
            encoded = tokenizer(chats[i:i + BATCH_SIZE], return_tensors="pt",
                                padding=True).to(model.device)
            hidden_states = model(**encoded, output_hidden_states=True).hidden_states
            lengths = encoded["attention_mask"].sum(dim=1)
            row_idx = torch.arange(lengths.size(0), device=lengths.device)
            for L in layers:
                out[L].append(hidden_states[L].float()[row_idx, lengths - 1].cpu())
        return {L: torch.cat(v) for L, v in out.items()}

    @torch.no_grad()
    def generate(chats):
        tokenizer.padding_side = "left"
        encoded = tokenizer(chats, return_tensors="pt", padding=True).to(model.device)
        # Vendor-recommended no-think sampling for Qwen3 (greedy is advised against).
        output = model.generate(
            **encoded, max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
        completions = output[:, encoded["input_ids"].shape[1]:]
        return tokenizer.batch_decode(completions, skip_special_tokens=True)

    if not ACT_CACHE.exists():
        acts = collect_final_token(chats, LAYERS)
        torch.save(acts, ACT_CACHE)
        print(f"saved activations -> {ACT_CACHE.name}")
    else:
        acts = torch.load(ACT_CACHE)

    if not rows:
        torch.manual_seed(SEED)
        with ROWS_PATH.open("w") as fh:
            for i in tqdm(range(0, len(chats), BATCH_SIZE), desc="generate"):
                responses = generate(chats[i:i + BATCH_SIZE])
                for record, response in zip(records[i:i + BATCH_SIZE], responses):
                    verdict = grade(response, record["metadata"])
                    row = {
                        "pair_id": record["pair_id"],
                        "theme": record["theme"],
                        "ops_total": record["ops_total"],
                        "status": verdict["status"],
                        "transform": verdict["transform"],
                        "response": response,
                    }
                    fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        rows = load_rows(ROWS_PATH)
        print(f"saved {len(rows)} graded rows -> {ROWS_PATH.name}")

assert [r["pair_id"] for r in rows] == [r["pair_id"] for r in records], "row/record misalignment"
status = np.array([r["status"] for r in rows])
for name in ("correct", "wrong", "unparseable"):
    print(f"{name:>12}: {(status == name).mean():.1%}")

## The two axes, and the cross-tab

**Axis 1, graded rather than binary.** For each prompt, rank the *other-theme* stories by cosine
similarity to it and record the rank at which the first same-law story appears. Rank 1 means the
model's representation of this story is closest to the same law told in a different setting —
"the law is represented". Higher ranks are a graded measure of how legible the law is, which
gives a dose–response curve instead of a single threshold.

A second, stricter variant restricts the candidate pool to stories with the **same total
operation count**, so structural size carries no information (the shape control from
experiment 3, section E, applied per example).

**Axis 2** is the graded verdict: `correct` / `wrong` / `unparseable`.

**Reading the table.** The two failure cells are the point: *represented but wrong* is a
formalization bottleneck, *not represented and wrong* is an abstraction bottleneck. Because both
axes worsen with complexity, the same comparison is repeated inside each operation-count bin and
a complexity-matched difference is reported — if the pooled association survives stratification
it is not merely "hard problems are hard".

**Kill criteria.** If accuracy is near 0% or near 100%, there is no variance to explain and the
cross-tab is uninformative. If `unparseable` dominates, the model is failing at output format
rather than at formalization, and that must be separated out before any bottleneck claim.

In [ ]:
# ---------------------- Axis 1: is the law represented? ----------------------

pair_ids = np.array([r["pair_id"] for r in records])
themes = np.array([r["theme"] for r in records])
ops = np.array([r["ops_total"] for r in records])
correct = status == "correct"

# Candidate pools: always a story of a *different* theme; the strict variant also
# demands the same operation count, so shape cannot be the cue.
POOLS = {
    "cross-theme": themes[:, None] != themes[None, :],
    "cross-theme, same ops": (themes[:, None] != themes[None, :]) & (ops[:, None] == ops[None, :]),
}


def law_ranks(X, allowed):
    """Rank of the first same-law candidate, per query (1 = nearest neighbour)."""
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    sims = Xn @ Xn.T
    sims[~allowed] = -np.inf
    ranks = np.zeros(len(X), dtype=int)
    for i, order in enumerate(np.argsort(-sims, axis=1)):
        order = order[np.isfinite(sims[i, order])]
        same = np.nonzero(pair_ids[order] == pair_ids[i])[0]
        ranks[i] = same[0] + 1 if len(same) else 0  # 0 = no same-law candidate in pool
    return ranks


ranks = {(name, L): law_ranks(acts[L].numpy(), allowed)
         for name, allowed in POOLS.items() for L in LAYERS}

print("share of prompts whose nearest other-theme story shares the law (rank 1)")
print(f"{'layer':>5}   " + "".join(f"{name:>24}" for name in POOLS))
for L in LAYERS:
    print(f"{L:>5}   " + "".join(f"{(ranks[(name, L)] == 1).mean():>24.1%}" for name in POOLS))

# ------------------------------- The cross-tab -------------------------------

VERDICTS = ("correct", "wrong", "unparseable")


def crosstab(represented):
    counts = {
        (rep, verdict): int(((represented == rep) & (status == verdict)).sum())
        for rep in (True, False) for verdict in VERDICTS
    }
    print(f"{'':>18}" + "".join(f"{v:>14}" for v in VERDICTS) + f"{'accuracy':>12}")
    for rep, label in ((True, "represented"), (False, "not represented")):
        total = sum(counts[(rep, v)] for v in VERDICTS)
        accuracy = counts[(rep, "correct")] / total if total else float("nan")
        print(f"{label:>18}" + "".join(f"{counts[(rep, v)]:>14}" for v in VERDICTS)
              + f"{accuracy:>12.1%}")
    return counts


def matched_difference(represented):
    """Accuracy gap between represented and not, averaged within complexity bins."""
    diffs, weights = [], []
    for bin_value in sorted(set(ops)):
        in_bin = ops == bin_value
        yes, no = in_bin & represented, in_bin & ~represented
        if yes.sum() and no.sum():
            diffs.append(correct[yes].mean() - correct[no].mean())
            weights.append(int(in_bin.sum()))
    if not diffs:
        return float("nan"), 0
    return float(np.average(diffs, weights=weights)), len(diffs)


for name in POOLS:
    represented = ranks[(name, PRIMARY_LAYER)] == 1
    print(f"\n=== pool: {name}, layer {PRIMARY_LAYER} ===")
    counts = crosstab(represented)
    pooled = (counts[(True, "correct")] / max(sum(counts[(True, v)] for v in VERDICTS), 1)
              - counts[(False, "correct")] / max(sum(counts[(False, v)] for v in VERDICTS), 1))
    matched, n_bins = matched_difference(represented)
    print(f"accuracy gap: pooled {pooled:+.1%}, complexity-matched {matched:+.1%} "
          f"({n_bins} bins contributed)")
    failures = ~correct
    if failures.sum():
        formalization = (failures & represented).sum() / failures.sum()
        print(f"of {int(failures.sum())} failures: {formalization:.0%} had the law represented "
              f"(formalization bottleneck), {1 - formalization:.0%} did not (abstraction bottleneck)")

# Layer sweep: does the conclusion depend on where we read?
print(f"\ncomplexity-matched accuracy gap by layer (pool: cross-theme)")
for L in LAYERS:
    matched, _ = matched_difference(ranks[("cross-theme", L)] == 1)
    print(f"  layer {L:>2}: {matched:+.1%}")

In [ ]:
# ------------------------------ Figures ------------------------------

import matplotlib.pyplot as plt

represented = ranks[("cross-theme", PRIMARY_LAYER)] == 1
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5))

# (1) Verdict composition, represented vs not.
groups = [("represented", represented), ("not represented", ~represented)]
colors = {"correct": "#2a9d3f", "wrong": "#e07b7b", "unparseable": "#7b7be0"}
bottoms = np.zeros(len(groups))
for verdict in VERDICTS:
    fractions = np.array([
        (status[mask] == verdict).mean() if mask.sum() else 0.0 for _, mask in groups])
    ax1.bar([label for label, _ in groups], fractions, bottom=bottoms,
            color=colors[verdict], label=verdict)
    bottoms += fractions
ax1.set_ylabel("share of prompts")
ax1.set_title(f"Verdict by whether the law is represented\n(layer {PRIMARY_LAYER})")
ax1.legend(fontsize=8)

# (2) Dose-response: accuracy against how legible the law is.
rank_values = ranks[("cross-theme", PRIMARY_LAYER)]
buckets = [("1", rank_values == 1), ("2-3", (rank_values >= 2) & (rank_values <= 3)),
           ("4-10", (rank_values >= 4) & (rank_values <= 10)), ("11+", rank_values >= 11)]
labels = [label for label, mask in buckets if mask.sum()]
rates = [correct[mask].mean() for _, mask in buckets if mask.sum()]
sizes = [int(mask.sum()) for _, mask in buckets if mask.sum()]
ax2.bar(labels, rates, color="#4878a8")
for x, (rate, size) in enumerate(zip(rates, sizes)):
    ax2.annotate(f"n={size}", (x, rate), ha="center", va="bottom", fontsize=8)
ax2.axhline(correct.mean(), color="gray", linestyle="--", lw=1,
            label=f"overall {correct.mean():.0%}")
ax2.set_xlabel("rank of the first same-law story (lower = more legible)")
ax2.set_ylabel("accuracy")
ax2.set_title("Dose-response: law legibility vs accuracy")
ax2.legend(fontsize=8)

# (3) The complexity confound, made visible.
bins = sorted(set(ops))
width = 0.38
for offset, (label, mask, color) in enumerate(
        [("represented", represented, "#2a9d3f"), ("not represented", ~represented, "#e07b7b")]):
    xs, ys = [], []
    for x, bin_value in enumerate(bins):
        in_bin = (ops == bin_value) & mask
        if in_bin.sum():
            xs.append(x + (offset - 0.5) * width)
            ys.append(correct[in_bin].mean())
    ax3.bar(xs, ys, width=width, color=color, label=label)
ax3.set_xticks(range(len(bins)))
ax3.set_xticklabels(bins)
ax3.set_xlabel("total operation count")
ax3.set_ylabel("accuracy")
ax3.set_title("Same comparison, within complexity bins")
ax3.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Interpretation notes

Fill in after the run:

- **Baseline behaviour.** Overall correct / wrong / unparseable rates. Is there enough variance
  for the cross-tab to mean anything, and is `unparseable` small enough not to dominate?
- **Axis 1 on prompted inputs.** Rank-1 share per layer, and how it compares with experiment 3's
  87% on bare texts — the prompt template adds hundreds of shared tokens, so some drop is
  expected; a large drop would mean the template washes out the law signal.
- **The cross-tab.** Counts in the four cells, the pooled accuracy gap, and — the number that
  matters — the complexity-matched gap.
- **Failure split.** What fraction of failures had the law represented?

**Decision mapping:**

- *Failures are mostly "law represented but output wrong"* → the bottleneck is formalization, not
  abstraction. Steering "represent this as a math problem" (experiment 1) is then aimed at the
  wrong stage, and the interesting target moves downstream: grammar/format control, constrained
  decoding, or the RG end of the formality ladder.
- *Failures are mostly "law not represented"* → abstraction is the bottleneck, which is exactly
  what experiment 1's steering and experiment 7's manifold intervention are designed to fix, and
  this notebook supplies the per-example labels to check whether steering rescues those cases.
- *No accuracy gap once complexity is matched* → the representation is not (visibly) used;
  retrieval quality and correctness are independently driven by problem size. That is a genuine
  negative result and it weakens the case for reading experiment 3's geometry as functional.

**Limits to state plainly.** This is still correlational — a gap shows the representation
*predicts* behaviour, not that the model reads from it. The causal version is activation
patching: transplant the law-identity content of story A into story B and see whether the output
follows. That, plus training a probe on these labels to get a pre-generation faithfulness
monitor, are the natural follow-ups.